# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">10/6(화) 오전 · LLM · 프롬프트 — 실습</mark>

지금까지 만든 룰은 우리가 정한 조건에 맞는 로그만 찾았습니다.

오늘은 LLM 에게 로그를 읽히고 **무슨 일인지 문장으로** 받습니다. 호출하고 → 지시를 다듬고 → 답을 JSON 으로 받습니다.

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기</mark>

### 0.1 맨 먼저 · 이 파일을 놓는 자리

1. 단톡방에서 받은 이 파일을 `security-agent-toolkit` 폴더 안의 **`agent_core` 폴더**에 넣습니다. `agent_core` 폴더가 없으면 만듭니다.
2. VS Code 에서 `security-agent-toolkit` 폴더를 열고, 왼쪽 목록의 `agent_core` 에서 이 파일을 엽니다.
3. 오른쪽 위 **커널 선택**을 눌러 Python 을 고릅니다.
4. 셀 실행은 **Shift + Enter** 입니다.

### 0.2 오늘 오전의 순서

| 교시 | 무엇 |
|---|---|
| 1교시 | 아침 과제 5 — Gemini API 키 발급 · `.env` · `.gitignore` |
| 2교시 | LLM 을 API 로 호출한다 — 주소 · 헤더 · 본문 · 응답에서 답 꺼내기 |
| 3교시 | 프롬프트 — 역할 · 지시 · 출력 형식 · 예시 |
| 4교시 | JSON 으로 받기 — `llm_client.py` |

### 0.3 막혔을 때

1. 문제 아래 **💡 힌트**를 순서대로 따라 합니다.
2. 그래도 막히면 노트북 맨 아래 「정답」으로 갑니다.
3. 도전 문제는 안 풀고 넘어가도 됩니다.

폴더는 이런 모양이 됩니다.

```
security-agent-toolkit/
  .env                ← 아침 과제에서 만든 키 파일
  .gitignore
  agent_core/
    01_1006_am_llm_prompt.ipynb   ← 이 파일
    llm_client.py                 ← 4교시에 여기에 만듭니다
  docs/               ← 그날 기록
```

### 0.4 문제를 푸는 법

1. 코드 셀의 번호 주석이 **무엇을 쓸지** 알려 줍니다. 주석 바로 아래 빈 줄에 코드를 씁니다.
2. 주석에 적힌 **변수 이름을 그대로** 씁니다. 아래 `print` 줄이 그 이름을 쓰기 때문입니다.
3. 앞 문제에서 쓴 줄을 뒤 문제에서 **다시 씁니다.** 같은 줄을 여러 번 쳐 보는 것이 연습입니다.
4. 미리 채워 둔 줄은 앞 날짜에 배운 것입니다. 고치지 않습니다.

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">2교시 (10:00–10:50) · LLM 을 API 로 호출한다</mark>

아래 두 셀을 먼저 실행합니다. 첫 셀은 `requests` 를 설치하고, 둘째 셀은 `.env` 가 제자리에 있는지 확인합니다.

이 노트북은 `agent_core` 폴더에 있고 `.env` 는 한 칸 위 `security-agent-toolkit` 폴더에 있습니다. 그래서 코드에서는 **`../.env`** 로 읽습니다. `..` 은 9/29 아침 과제에서 쓴 「한 칸 위 폴더」입니다.

In [ ]:
%pip install -q requests

In [ ]:
import os

if os.path.exists("../.env"):
    print(".env 파일이 있습니다. 2교시를 시작합니다.")
else:
    print(".env 파일이 없습니다. 아침 과제 5번을 먼저 끝내고, 이 노트북이 agent_core 폴더 안에 있는지 확인합니다.")

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 상관없습니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **토큰 (token)** | LLM 이 글을 어떤 단위로 잘라서 세는가 |
| **환각 (hallucination)** | LLM 이 틀린 답을 낼 때 어떤 모양으로 틀리는가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- 토큰 →
- 환각 →

적어 둔 것은 2교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.

---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2 · LLM 을 API 로 호출한다</mark>

### 왜 필요한가

1. 지금까지 만든 룰은 **우리가 정한 조건**에 맞는 로그만 찾았습니다. 조건에 없는 로그는 그대로 지나갑니다.
2. LLM 은 로그 한 줄을 읽고 무슨 일인지 **문장으로 설명**합니다. 사람이 한 줄씩 읽던 일을 대신합니다.
3. LLM 을 쓰는 방법은 새로 배우지 않습니다. 10/2 에 배운 `requests.post` 로 요청을 보내고 JSON 응답을 받습니다.

### 이 시간에 나오는 말

| 말 | 뜻 | 2교시에서 보는 모양 |
|---|---|---|
| LLM | 다음에 올 낱말을 확률로 고르는 큰 모델 | `"model": "gemini-3.5-flash-lite"` |
| 토큰 | 모델이 글을 자르는 최소 단위. 사용량과 한도를 토큰으로 센다 | `"total_tokens": 50` |
| 환각 | 사실이 아닌 것을 그럴듯하게 지어내는 것 | LLM `SK텔레콤` ↔ 조회 API `SamsungSDS Inc.` |

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.1 LLM 의 응답은 딕셔너리로 옵니다</mark>

「한 문장으로 답하세요. 피싱이란 무엇입니까?」 라고 물었을 때 실제로 받은 응답입니다. 오늘 쓰지 않는 칸은 줄였습니다.

```python
data = {
    "choices": [
        {"message": {"role": "assistant",
                     "content": "피싱이란 신뢰할 수 있는 기관이나 사람으로 위장하여 타인의 개인정보나 금융정보를 속여 빼내는 전자 금융 사기 수법입니다."}}
    ],
    "model": "gemini-3.5-flash-lite",
    "usage": {"prompt_tokens": 15, "completion_tokens": 35, "total_tokens": 50}
}

print(data["choices"][0]["message"]["content"])
```

1. 답 문장은 `choices` → `0` → `message` → `content` 순서로 들어가야 나옵니다.
2. `choices` 는 **리스트**라서 숫자 `0` 으로 꺼냅니다. `message` 와 `content` 는 **딕셔너리의 키**라서 이름으로 꺼냅니다.
3. `usage` 에는 토큰 수가 있습니다. 질문이 15토큰, 답이 35토큰, 합계가 50토큰입니다.

9/28 오후에 배운 중첩 자료와 같은 방법입니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
data = {
    "choices": [
        {"message": {"role": "assistant",
                     "content": "포트 스캔이란 컴퓨터 네트워크의 취약점을 찾기 위해 특정 서버의 열려 있는 포트와 활성화된 서비스 탐색을 시도하는 해킹 또는 보안 진단 기법입니다."}}
    ],
    "usage": {"prompt_tokens": 17, "completion_tokens": 43, "total_tokens": 60}
}

print(data["choices"][0]["message"]["role"])
```

막히면 바로 위 `2.1 LLM 의 응답은 딕셔너리로 옵니다` 설명을 다시 봅니다.

In [ ]:
data = {
    "choices": [
        {"message": {"role": "assistant",
                     "content": "포트 스캔이란 컴퓨터 네트워크의 취약점을 찾기 위해 특정 서버의 열려 있는 포트와 활성화된 서비스 탐색을 시도하는 해킹 또는 보안 진단 기법입니다."}}
    ],
    "usage": {"prompt_tokens": 17, "completion_tokens": 43, "total_tokens": 60}
}

print(data["choices"][0]["message"]["role"])

✅ `assistant`

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-2 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
data = {
    "choices": [
        {"message": {"role": "assistant",
                     "content": "포트 스캔이란 컴퓨터 네트워크의 취약점을 찾기 위해 특정 서버의 열려 있는 포트와 활성화된 서비스 탐색을 시도하는 해킹 또는 보안 진단 기법입니다."}}
    ],
    "usage": {"prompt_tokens": 17, "completion_tokens": 43, "total_tokens": 60}
}

usage = data["usage"]
print(usage["prompt_tokens"] + usage["completion_tokens"])
```

막히면 바로 위 `2.1 LLM 의 응답은 딕셔너리로 옵니다` 설명을 다시 봅니다.

In [ ]:
data = {
    "choices": [
        {"message": {"role": "assistant",
                     "content": "포트 스캔이란 컴퓨터 네트워크의 취약점을 찾기 위해 특정 서버의 열려 있는 포트와 활성화된 서비스 탐색을 시도하는 해킹 또는 보안 진단 기법입니다."}}
    ],
    "usage": {"prompt_tokens": 17, "completion_tokens": 43, "total_tokens": 60}
}

usage = data["usage"]
print(usage["prompt_tokens"] + usage["completion_tokens"])

✅ `60` — 질문 토큰과 답 토큰을 더하면 `total_tokens` 와 같습니다.

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.2 호출에는 주소 · 헤더 · 본문이 필요합니다</mark>

9/30 과 10/2 에 쓴 `requests` 그대로입니다. 넣는 값만 다릅니다.

```python
import requests

api_key = "발급받은_키"      # 실제로는 .env 에서 읽습니다

url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}
body = {
    "model": "gemini-3.5-flash-lite",
    "messages": [{"role": "user", "content": "한 문장으로 답하세요. 피싱이란 무엇입니까?"}]
}

response = requests.post(url, headers=headers, json=body, timeout=30)
print(response.status_code)          # 200
```

1. **주소** — LLM 서비스가 요청을 받는 곳입니다.
2. **헤더** — `Authorization` 에 `Bearer` 와 빈칸 한 칸, 그 뒤에 키를 붙입니다. 9/29 오후에 배운 인증 방식입니다.
3. **본문** — `model` 에는 모델 이름을, `messages` 에는 질문을 넣습니다. 질문은 `role` 이 `"user"` 인 딕셔너리 하나입니다.
4. 무료 등급은 **1분에 15번**까지 호출할 수 있습니다. 넘으면 상태코드 `429` 가 오고, 1분쯤 기다리면 다시 됩니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-3 · 헤더 만들기</font></h3></td></tr></table>

`.env` 에서 읽은 키로 요청 헤더를 만드시오. **키는 화면에 출력하지 않습니다.**

| | |
|---|---|
| 🎯 나와야 하는 결과 | `7` |

**💡 힌트**

1. 헤더는 딕셔너리입니다. 키는 `Authorization` 과 `Content-Type` 둘입니다.
2. `Authorization` 값은 글자 두 개를 `+` 로 이어 붙입니다. `Bearer` 뒤에 빈칸이 한 칸 있어야 합니다.
3. 결과 `7` 은 `Bearer` 여섯 글자에 빈칸 한 칸을 더한 수입니다. 키 길이를 뺐기 때문에 키는 드러나지 않습니다.

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = None
with open("../.env", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("=", 1)
        if parts[0] == "GEMINI_API_KEY":
            api_key = parts[1]
# ── 여기까지 ──

# 1. headers 라는 변수를 만들어 "Bearer " 와 api_key 를 이어 붙인 헤더를 담으세요
#    Content-Type 은 "application/json" 입니다


print(len(headers["Authorization"]) - len(api_key))

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-4 · 본문 만들기</font></h3></td></tr></table>

LLM 에게 보낼 본문을 만드시오. 질문은 변수 `question` 에 들어 있습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `user` 그리고 `1` |

**💡 힌트**

1. 본문은 딕셔너리입니다. 키는 `model` 과 `messages` 둘입니다.
2. `messages` 는 리스트이고, 그 안에 딕셔너리 하나를 넣습니다. 키는 `role` 과 `content` 입니다.
3. `content` 값에는 글자를 직접 쓰지 않고 변수 `question` 을 넣습니다.

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
question = "다음 로그 한 줄이 무슨 일인지 한 문장으로 설명하세요. 로그: 2026-10-06 02:41:17 WARN failed login for svc_batch from 198.51.100.7"
# ── 여기까지 ──

# 1. body 라는 변수를 만들어 본문을 담으세요
#    모델 이름은 "gemini-3.5-flash-lite", 질문의 role 은 "user" 입니다


print(body["messages"][0]["role"])
print(len(body["messages"]))

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-5 · LLM 을 처음 호출하기</font></h3></td></tr></table>

헤더와 본문을 만들고 LLM 에 요청을 보낸 뒤, 상태코드를 출력하시오. 문제 2-3 과 2-4 에서 쓴 줄을 다시 씁니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `200` |

**💡 힌트**

1. 첫째 줄과 둘째 줄은 문제 2-3, 2-4 에서 만든 것과 같습니다.
2. 셋째 줄은 10/2 오후에 쓴 `requests.post` 와 같은 모양입니다. 괄호 안에 주소, `headers=`, `json=`, `timeout=30` 을 넣습니다.
3. `200` 이 아니면 아침 과제에서 만든 `.env` 의 키를 다시 확인합니다.

In [ ]:
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = None
with open("../.env", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("=", 1)
        if parts[0] == "GEMINI_API_KEY":
            api_key = parts[1]
url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
question = "다음 로그 한 줄이 무슨 일인지 한 문장으로 설명하세요. 로그: 2026-10-06 02:41:17 WARN failed login for svc_batch from 198.51.100.7"
# ── 여기까지 ──

# 1. headers 라는 변수를 만들어 "Bearer " 와 api_key 를 이어 붙인 헤더를 담으세요


# 2. body 라는 변수를 만들어 모델 이름과 question 을 넣은 본문을 담으세요


# 3. response 라는 변수를 만들어 url 로 보낸 POST 요청의 결과를 담으세요


print(response.status_code)

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-6 · 응답에서 답 문장만 꺼내기</font></h3></td></tr></table>

LLM 에 요청을 보내고, 응답에서 **답 문장만** 꺼내 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `198.51.100.7 아이피에서 svc_batch 계정으로 로그인을 시도했으나 실패했습니다.` 와 비슷한 한 문장. 문장은 실행할 때마다 조금 달라질 수 있습니다 |

**💡 힌트**

1. 본문 만들기와 요청 보내기는 문제 2-5 와 같습니다.
2. 응답을 딕셔너리로 바꾸는 방법은 9/30 에 배운 `.json()` 입니다.
3. 답 문장은 바깥에서 안으로 한 칸씩 들어가 꺼냅니다 — `choices`, 0번, `message`, `content` 순서입니다.

In [ ]:
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = None
with open("../.env", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("=", 1)
        if parts[0] == "GEMINI_API_KEY":
            api_key = parts[1]
url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}
question = "다음 로그 한 줄이 무슨 일인지 한 문장으로 설명하세요. 로그: 2026-10-06 02:41:17 WARN failed login for svc_batch from 198.51.100.7"
# ── 여기까지 ──

# 1. body 라는 변수를 만들어 모델 이름과 question 을 넣은 본문을 담으세요


# 2. response 라는 변수를 만들어 url 로 보낸 POST 요청의 결과를 담으세요


# 3. data 라는 변수를 만들어 응답을 딕셔너리로 바꾼 것을 담으세요


# 4. answer 라는 변수를 만들어 data 에서 꺼낸 답 문장을 담으세요


print(answer)

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-7 · 토큰을 몇 개 썼는지 보기</font></h3></td></tr></table>

다른 로그를 물어보고, 요청 한 번에 쓴 **토큰 합계**를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `쓴 토큰: 101` 처럼 숫자 하나. 답 문장이 길어지면 숫자도 커집니다 (실제로 101 · 116 · 126 이 나왔습니다) |

**💡 힌트**

1. 본문, 요청, 딕셔너리로 바꾸기는 문제 2-6 과 같습니다.
2. 토큰 수는 `usage` 안에 있습니다. 세 값 가운데 합계를 꺼냅니다.
3. 이 질문의 질문 토큰은 56 으로 늘 같습니다. 달라지는 것은 답 토큰입니다.

In [ ]:
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = None
with open("../.env", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("=", 1)
        if parts[0] == "GEMINI_API_KEY":
            api_key = parts[1]
url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}
question = "다음 로그 한 줄이 위험한지 한 문장으로 답하세요. 로그: 2026-10-06 04:12:55 WARN failed login for root from 192.0.2.44"
# ── 여기까지 ──

# 1. body 라는 변수를 만들어 모델 이름과 question 을 넣은 본문을 담으세요


# 2. response 라는 변수를 만들어 url 로 보낸 POST 요청의 결과를 담으세요


# 3. data 라는 변수를 만들어 응답을 딕셔너리로 바꾼 것을 담으세요


# 4. total 이라는 변수를 만들어 data 에서 꺼낸 토큰 합계를 담으세요


print("쓴 토큰:", total)

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.3 LLM 은 조회하지 않고 문장을 만듭니다</mark>

LLM 은 어딘가를 찾아보고 답하지 않습니다. 그럴듯한 낱말을 이어 붙여 문장을 만듭니다. 그래서 **모르는 것도 아는 것처럼 답합니다.** 이것을 환각이라고 합니다.

같은 질문 「IP 211.45.12.9 는 어느 통신사의 주소입니까?」 에 실제로 받은 답입니다.

| 누구에게 물었나 | 받은 답 |
|---|---|
| LLM · 첫 번째 | `LG Uplus` |
| LLM · 두 번째 | `SK텔레콤` |
| 조회 API `ipwho.is` (9/30 에 쓴 서비스) | `SamsungSDS Inc.` |

1. LLM 의 답은 **물을 때마다 달라졌고, 둘 다 틀렸습니다.** 문장은 자연스러워서 틀린 줄 알기 어렵습니다.
2. 조회 API 는 실제 등록 정보를 찾아서 돌려줍니다.
3. 그래서 **사실 확인은 조회 API 와 로그로** 하고, LLM 에게는 요약과 설명을 맡깁니다. 최종 판단은 사람이 합니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-8 · LLM 의 답과 조회 API 의 답 견주기</font></h3></td></tr></table>

IP `211.45.12.9` 의 통신사를 LLM 과 조회 API 양쪽에 묻습니다. LLM 의 답 문장과, 조회 API 응답 `truth` 에 든 **통신사 이름**을 나란히 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `LLM     : SK텔레콤` (사람마다 다릅니다) 그리고 `조회 API : SamsungSDS Inc.` |

**💡 힌트**

1. LLM 쪽 세 줄은 문제 2-6 과 같습니다.
2. `truth` 는 딕셔너리 안에 딕셔너리가 든 모양입니다. 통신사 이름은 `connection` 안의 `isp` 칸에 있습니다.
3. 두 줄이 다르면 LLM 이 지어낸 것입니다. 조회 API 쪽이 실제 값입니다.

In [ ]:
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = None
with open("../.env", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("=", 1)
        if parts[0] == "GEMINI_API_KEY":
            api_key = parts[1]
url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}
ip = "211.45.12.9"
question = "IP " + ip + " 는 어느 통신사의 주소입니까? 통신사 이름만 답하세요."
body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": question}]}
truth = requests.get("https://ipwho.is/" + ip, timeout=10).json()
# ── 여기까지 ──

# 1. response 라는 변수를 만들어 url 로 보낸 POST 요청의 결과를 담으세요


# 2. data 라는 변수를 만들어 응답을 딕셔너리로 바꾼 것을 담으세요


# 3. answer 라는 변수를 만들어 data 에서 꺼낸 답 문장을 담으세요


# 4. isp 라는 변수를 만들어 truth 에서 꺼낸 통신사 이름을 담으세요


print("LLM     :", answer)
print("조회 API :", isp)

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-9 · 키가 틀렸을 때를 대비하기</font></h3></td></tr></table>

일부러 **틀린 키**로 호출했습니다. 요청이 성공했으면 답 문장을, 실패했으면 `실패:` 와 상태코드를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `실패: 400` |

**💡 힌트**

1. 성공인지 확인하는 기준은 상태코드입니다. 성공일 때의 숫자는 9/29 오후에 배웠습니다.
2. 성공했을 때 할 일은 문제 2-6 의 뒤쪽 두 줄과 같습니다.
3. 실패한 응답에는 `choices` 칸이 없습니다. 확인 없이 꺼내면 프로그램이 멈춥니다.

In [ ]:
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = "wrong-key-1234"      # 일부러 틀린 키입니다. 가짜 키라서 코드에 적어도 됩니다
url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}
question = "다음 로그 한 줄이 무슨 일인지 한 문장으로 설명하세요. 로그: 2026-10-06 02:41:17 WARN failed login for svc_batch from 198.51.100.7"
body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": question}]}
response = requests.post(url, headers=headers, json=body, timeout=30)
# ── 여기까지 ──

# 1. response 의 상태코드가 200 이면
#    data 라는 변수에 응답을 딕셔너리로 바꿔 담고, 답 문장을 출력하세요
# 2. 200 이 아니면 "실패:" 와 상태코드를 출력하세요

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 2-10 · 로그 세 줄을 차례로 설명시키기</font></h3></td></tr></table>

`logs` 의 로그 세 줄을 **한 줄씩** LLM 에 보내고, 답 문장을 한 줄씩 출력하시오. 새 문법은 없습니다. 문제 2-5 · 2-6 · 2-9 를 반복문 안에 넣습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 문장 세 줄. 첫 줄은 `jdoe` 의 로그인 성공, 둘째 줄은 `guest01` 의 권한 거부, 셋째 줄은 `deploy` 의 토큰 만료를 설명합니다 |

**💡 힌트**

1. 헤더는 반복 밖에서 한 번만 만듭니다. 본문은 로그마다 달라지므로 반복 안에서 만듭니다.
2. 질문 글자는 `"한 문장으로 설명하세요. 로그: " + log` 로 만듭니다.
3. 답을 꺼내기 전에 상태코드를 확인하는 것은 문제 2-9 와 같습니다.

In [ ]:
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = None
with open("../.env", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("=", 1)
        if parts[0] == "GEMINI_API_KEY":
            api_key = parts[1]
url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
logs = [
    "2026-10-06 09:14:52 INFO accepted login for jdoe from 192.0.2.44",
    "2026-10-06 13:07:30 WARN permission denied for guest01 on /etc/shadow",
    "2026-10-06 23:58:11 WARN token expired for deploy from 203.0.113.61",
]
# ── 여기까지 ──

# 1. headers 라는 변수를 만들어 헤더를 담으세요
# 2. logs 를 하나씩 꺼내 반복하세요. 꺼낸 로그의 변수 이름은 log 입니다
# 3. 반복 안에서 body 와 response 를 만들어 요청을 보내세요
# 4. 상태코드가 200 이면 답 문장을 출력하세요
# 5. 200 이 아니면 "실패:" 와 상태코드를 출력하세요

### 📋 2교시 한눈에

| 하려는 일 | 쓰는 코드 |
|---|---|
| LLM 에 요청을 보낸다 | `requests.post(url, headers=headers, json=body, timeout=30)` |
| 키를 싣는다 | `{"Authorization": "Bearer " + api_key}` |
| 질문을 담는다 | `"messages": [{"role": "user", "content": question}]` |
| 답 문장을 꺼낸다 | `data["choices"][0]["message"]["content"]` |
| 쓴 토큰을 본다 | `data["usage"]["total_tokens"]` |
| 성공했는지 확인한다 | `response.status_code == 200` |

LLM 의 답은 문장이 자연스러워도 틀릴 수 있습니다. 사실 확인은 조회 API 와 로그로 합니다.

---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 봅니다</mark>

출력 예상 문제(2-1 · 2-2)의 답은 문제 바로 아래 ✅ 줄에 있습니다.

### 정답 2-3

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = None
with open("../.env", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("=", 1)
        if parts[0] == "GEMINI_API_KEY":
            api_key = parts[1]
# ── 여기까지 ──

# 1. headers 라는 변수를 만들어 "Bearer " 와 api_key 를 이어 붙인 헤더를 담으세요
#    Content-Type 은 "application/json" 입니다
headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}

print(len(headers["Authorization"]) - len(api_key))

### 정답 2-4

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
question = "다음 로그 한 줄이 무슨 일인지 한 문장으로 설명하세요. 로그: 2026-10-06 02:41:17 WARN failed login for svc_batch from 198.51.100.7"
# ── 여기까지 ──

# 1. body 라는 변수를 만들어 본문을 담으세요
#    모델 이름은 "gemini-3.5-flash-lite", 질문의 role 은 "user" 입니다
body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": question}]}

print(body["messages"][0]["role"])
print(len(body["messages"]))

### 정답 2-5

In [ ]:
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = None
with open("../.env", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("=", 1)
        if parts[0] == "GEMINI_API_KEY":
            api_key = parts[1]
url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
question = "다음 로그 한 줄이 무슨 일인지 한 문장으로 설명하세요. 로그: 2026-10-06 02:41:17 WARN failed login for svc_batch from 198.51.100.7"
# ── 여기까지 ──

# 1. headers 라는 변수를 만들어 "Bearer " 와 api_key 를 이어 붙인 헤더를 담으세요
headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}

# 2. body 라는 변수를 만들어 모델 이름과 question 을 넣은 본문을 담으세요
body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": question}]}

# 3. response 라는 변수를 만들어 url 로 보낸 POST 요청의 결과를 담으세요
response = requests.post(url, headers=headers, json=body, timeout=30)

print(response.status_code)

### 정답 2-6

In [ ]:
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = None
with open("../.env", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("=", 1)
        if parts[0] == "GEMINI_API_KEY":
            api_key = parts[1]
url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}
question = "다음 로그 한 줄이 무슨 일인지 한 문장으로 설명하세요. 로그: 2026-10-06 02:41:17 WARN failed login for svc_batch from 198.51.100.7"
# ── 여기까지 ──

# 1. body 라는 변수를 만들어 모델 이름과 question 을 넣은 본문을 담으세요
body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": question}]}

# 2. response 라는 변수를 만들어 url 로 보낸 POST 요청의 결과를 담으세요
response = requests.post(url, headers=headers, json=body, timeout=30)

# 3. data 라는 변수를 만들어 응답을 딕셔너리로 바꾼 것을 담으세요
data = response.json()

# 4. answer 라는 변수를 만들어 data 에서 꺼낸 답 문장을 담으세요
answer = data["choices"][0]["message"]["content"]

print(answer)

### 정답 2-7

In [ ]:
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = None
with open("../.env", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("=", 1)
        if parts[0] == "GEMINI_API_KEY":
            api_key = parts[1]
url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}
question = "다음 로그 한 줄이 위험한지 한 문장으로 답하세요. 로그: 2026-10-06 04:12:55 WARN failed login for root from 192.0.2.44"
# ── 여기까지 ──

# 1. body 라는 변수를 만들어 모델 이름과 question 을 넣은 본문을 담으세요
body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": question}]}

# 2. response 라는 변수를 만들어 url 로 보낸 POST 요청의 결과를 담으세요
response = requests.post(url, headers=headers, json=body, timeout=30)

# 3. data 라는 변수를 만들어 응답을 딕셔너리로 바꾼 것을 담으세요
data = response.json()

# 4. total 이라는 변수를 만들어 data 에서 꺼낸 토큰 합계를 담으세요
total = data["usage"]["total_tokens"]

print("쓴 토큰:", total)

### 정답 2-8

In [ ]:
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = None
with open("../.env", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("=", 1)
        if parts[0] == "GEMINI_API_KEY":
            api_key = parts[1]
url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}
ip = "211.45.12.9"
question = "IP " + ip + " 는 어느 통신사의 주소입니까? 통신사 이름만 답하세요."
body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": question}]}
truth = requests.get("https://ipwho.is/" + ip, timeout=10).json()
# ── 여기까지 ──

# 1. response 라는 변수를 만들어 url 로 보낸 POST 요청의 결과를 담으세요
response = requests.post(url, headers=headers, json=body, timeout=30)

# 2. data 라는 변수를 만들어 응답을 딕셔너리로 바꾼 것을 담으세요
data = response.json()

# 3. answer 라는 변수를 만들어 data 에서 꺼낸 답 문장을 담으세요
answer = data["choices"][0]["message"]["content"]

# 4. isp 라는 변수를 만들어 truth 에서 꺼낸 통신사 이름을 담으세요
isp = truth["connection"]["isp"]

print("LLM     :", answer)
print("조회 API :", isp)

### 정답 2-9

In [ ]:
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = "wrong-key-1234"      # 일부러 틀린 키입니다. 가짜 키라서 코드에 적어도 됩니다
url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}
question = "다음 로그 한 줄이 무슨 일인지 한 문장으로 설명하세요. 로그: 2026-10-06 02:41:17 WARN failed login for svc_batch from 198.51.100.7"
body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": question}]}
response = requests.post(url, headers=headers, json=body, timeout=30)
# ── 여기까지 ──

# 1. response 의 상태코드가 200 이면
#    data 라는 변수에 응답을 딕셔너리로 바꿔 담고, 답 문장을 출력하세요
# 2. 200 이 아니면 "실패:" 와 상태코드를 출력하세요
if response.status_code == 200:
    data = response.json()
    print(data["choices"][0]["message"]["content"])
else:
    print("실패:", response.status_code)

### 정답 2-10

In [ ]:
import requests

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
api_key = None
with open("../.env", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("=", 1)
        if parts[0] == "GEMINI_API_KEY":
            api_key = parts[1]
url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
logs = [
    "2026-10-06 09:14:52 INFO accepted login for jdoe from 192.0.2.44",
    "2026-10-06 13:07:30 WARN permission denied for guest01 on /etc/shadow",
    "2026-10-06 23:58:11 WARN token expired for deploy from 203.0.113.61",
]
# ── 여기까지 ──

# 1. headers 라는 변수를 만들어 헤더를 담으세요
# 2. logs 를 하나씩 꺼내 반복하세요. 꺼낸 로그의 변수 이름은 log 입니다
# 3. 반복 안에서 body 와 response 를 만들어 요청을 보내세요
# 4. 상태코드가 200 이면 답 문장을 출력하세요
# 5. 200 이 아니면 "실패:" 와 상태코드를 출력하세요
headers = {"Authorization": "Bearer " + api_key, "Content-Type": "application/json"}
for log in logs:
    body = {"model": "gemini-3.5-flash-lite", "messages": [{"role": "user", "content": "한 문장으로 설명하세요. 로그: " + log}]}
    response = requests.post(url, headers=headers, json=body, timeout=30)
    if response.status_code == 200:
        data = response.json()
        print(data["choices"][0]["message"]["content"])
    else:
        print("실패:", response.status_code)